# Adaptive Model Serving Optimizer - Exploration Notebook

This notebook provides comprehensive exploration and analysis of the adaptive model serving optimization system. It demonstrates the key capabilities, performs comparative analysis of serving strategies, and provides insights into the multi-armed bandit optimization process.

## Table of Contents
1. [Setup and Configuration](#setup)
2. [Model Serving Adapters](#adapters)
3. [Multi-Armed Bandit Algorithms](#bandits)
4. [Performance Benchmarking](#benchmarking)
5. [Optimization Experiment](#experiment)
6. [Results Analysis](#analysis)
7. [Production Deployment Guide](#deployment)

## 1. Setup and Configuration {#setup}

Let's start by setting up the environment and configuring the system.

In [ ]:
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path().parent / 'src'))

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.models as models
import time
import json
from typing import Dict, List, Any

# Configure plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_palette("husl")

print("Environment setup complete!")

In [ ]:
# Import adaptive model serving optimizer components
from adaptive_model_serving_optimizer import (
    Config,
    get_config,
    setup_environment_from_config,
    ModelLoader,
    DatasetLoader,
    BenchmarkDataset,
    ModelOptimizer,
    PerformanceBenchmark,
    ModelAdapterFactory,
    PyTorchAdapter,
    ONNXAdapter,
    ServingStrategyOptimizer,
    UCBBandit,
    ThompsonSamplingBandit,
    EpsilonGreedyBandit,
    MetricsCollector,
    ModelDriftDetector
)

print("Adaptive Model Serving Optimizer imported successfully!")

In [ ]:
# Create and configure the system
config = Config()
config.device = 'cuda' if torch.cuda.is_available() else 'cpu'
config.training.batch_size = 16
config.bandits.algorithm = 'ucb'
config.bandits.epsilon = 0.1
config.debug = True

# Setup environment
setup_environment_from_config(config)

print(f"Configuration created - Device: {config.device}")
print(f"Bandit algorithm: {config.bandits.algorithm}")
print(f"Batch size: {config.training.batch_size}")

## 2. Model Serving Adapters {#adapters}

Let's explore the different model serving adapters and their capabilities.

In [ ]:
# Create sample models for demonstration
def create_sample_models():
    """Create sample models with different architectures."""
    models_dict = {}
    
    # ResNet-18 (smaller, faster)
    resnet18 = models.resnet18(pretrained=False, num_classes=10)
    resnet18.eval()
    models_dict['resnet18'] = resnet18
    
    # ResNet-50 (larger, more accurate)
    resnet50 = models.resnet50(pretrained=False, num_classes=10)
    resnet50.eval()
    models_dict['resnet50'] = resnet50
    
    # MobileNet (efficient)
    mobilenet = models.mobilenet_v2(pretrained=False, num_classes=10)
    mobilenet.eval()
    models_dict['mobilenet'] = mobilenet
    
    return models_dict

# Create models
sample_models = create_sample_models()

print("Created sample models:")
for name, model in sample_models.items():
    num_params = sum(p.numel() for p in model.parameters())
    print(f"  {name}: {num_params:,} parameters")

In [ ]:
# Create serving adapters with different configurations
def create_serving_adapters(models_dict: Dict[str, nn.Module], config: Config) -> Dict[str, PyTorchAdapter]:
    """Create serving adapters with different optimization levels."""
    adapters = {}
    
    for model_name, model in models_dict.items():
        # Standard configuration
        standard_config = Config(**config.__dict__)
        standard_config.serving.pytorch_config.update({
            'precision': 'float32',
            'jit_compile': False,
            'compile_mode': 'none'
        })
        adapters[f'{model_name}_standard'] = PyTorchAdapter(standard_config, model=model)
        
        # Optimized configuration
        optimized_config = Config(**config.__dict__)
        optimized_config.serving.pytorch_config.update({
            'precision': 'float16',
            'jit_compile': True,
            'compile_mode': 'reduce-overhead'
        })
        adapters[f'{model_name}_optimized'] = PyTorchAdapter(optimized_config, model=model.half())
    
    return adapters

# Create adapters
serving_adapters = create_serving_adapters(sample_models, config)

print(f"Created {len(serving_adapters)} serving adapters:")
for name in serving_adapters.keys():
    print(f"  {name}")

## 3. Multi-Armed Bandit Algorithms {#bandits}

Let's explore the different bandit algorithms and their characteristics.

In [ ]:
# Compare different bandit algorithms
def simulate_bandit_behavior(num_arms: int = 4, num_experiments: int = 1000):
    """Simulate and compare bandit algorithm behaviors."""
    
    # Create test configurations for different algorithms
    algorithms = {
        'UCB': UCBBandit,
        'Thompson Sampling': ThompsonSamplingBandit,
        'Epsilon-Greedy': EpsilonGreedyBandit
    }
    
    # Simulate true reward distributions for each arm
    true_rewards = [0.3, 0.7, 0.5, 0.9]  # Arm 3 is best, Arm 0 is worst
    arm_names = [f'arm_{i}' for i in range(num_arms)]
    
    results = {}
    
    for algo_name, algo_class in algorithms.items():
        bandit = algo_class(config)
        
        # Add arms
        for arm_name in arm_names:
            bandit.add_arm(arm_name)
        
        # Track selection history
        selections = []
        rewards = []
        regrets = []
        cumulative_regret = 0
        
        for experiment in range(num_experiments):
            # Select arm
            selected_arm = bandit.select_arm()
            arm_idx = arm_names.index(selected_arm)
            
            # Generate reward (Bernoulli with true probability)
            reward = np.random.binomial(1, true_rewards[arm_idx])
            
            # Calculate instantaneous regret
            best_reward = max(true_rewards)
            regret = best_reward - true_rewards[arm_idx]
            cumulative_regret += regret
            
            # Update bandit
            bandit.update_arm(
                selected_arm, 
                reward, 
                latency=10.0 + np.random.normal(0, 1),
                throughput=100.0 + np.random.normal(0, 5),
                error_rate=0.01
            )
            
            # Record
            selections.append(arm_idx)
            rewards.append(reward)
            regrets.append(cumulative_regret)
        
        results[algo_name] = {
            'selections': selections,
            'rewards': rewards,
            'regrets': regrets,
            'bandit': bandit
        }
    
    return results, true_rewards, arm_names

# Run simulation
bandit_results, true_rewards, arm_names = simulate_bandit_behavior()
print("Bandit algorithm simulation completed!")
print(f"True reward probabilities: {true_rewards}")
print(f"Best arm: arm_{np.argmax(true_rewards)} (reward: {max(true_rewards)})")

In [ ]:
# Visualize bandit algorithm performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Multi-Armed Bandit Algorithm Comparison', fontsize=16)

# Plot 1: Cumulative regret
ax1 = axes[0, 0]
for algo_name, data in bandit_results.items():
    ax1.plot(data['regrets'], label=algo_name, linewidth=2)
ax1.set_xlabel('Experiment Number')
ax1.set_ylabel('Cumulative Regret')
ax1.set_title('Cumulative Regret Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Arm selection frequency
ax2 = axes[0, 1]
width = 0.25
x = np.arange(len(arm_names))

for i, (algo_name, data) in enumerate(bandit_results.items()):
    selections = np.array(data['selections'])
    frequencies = [np.sum(selections == j) / len(selections) for j in range(len(arm_names))]
    ax2.bar(x + i*width, frequencies, width, label=algo_name, alpha=0.8)

ax2.set_xlabel('Arm')
ax2.set_ylabel('Selection Frequency')
ax2.set_title('Final Arm Selection Frequencies')
ax2.set_xticks(x + width)
ax2.set_xticklabels(arm_names)
ax2.legend()
ax2.grid(True, alpha=0.3)

# Add true reward line
ax2_twin = ax2.twinx()
ax2_twin.plot(x, true_rewards, 'ro-', linewidth=2, markersize=8, label='True Rewards')
ax2_twin.set_ylabel('True Reward Probability')
ax2_twin.legend(loc='upper right')

# Plot 3: Selection evolution over time
ax3 = axes[1, 0]
window_size = 100

for algo_name, data in bandit_results.items():
    selections = np.array(data['selections'])
    best_arm_idx = np.argmax(true_rewards)
    
    # Calculate rolling frequency of selecting best arm
    best_arm_freq = []
    for i in range(window_size, len(selections)):
        window = selections[i-window_size:i]
        freq = np.sum(window == best_arm_idx) / window_size
        best_arm_freq.append(freq)
    
    x_axis = range(window_size, len(selections))
    ax3.plot(x_axis, best_arm_freq, label=f'{algo_name}', linewidth=2)

ax3.set_xlabel('Experiment Number')
ax3.set_ylabel('Best Arm Selection Frequency')
ax3.set_title(f'Best Arm Selection Frequency (Window: {window_size})')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='Optimal')

# Plot 4: Final arm statistics
ax4 = axes[1, 1]
algo_names = list(bandit_results.keys())
final_regrets = [data['regrets'][-1] for data in bandit_results.values()]

bars = ax4.bar(algo_names, final_regrets, alpha=0.7, color=['blue', 'orange', 'green'])
ax4.set_ylabel('Final Cumulative Regret')
ax4.set_title('Final Performance Comparison')
ax4.grid(True, alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, final_regrets):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
             f'{value:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 4. Performance Benchmarking {#benchmarking}

Let's benchmark the performance of different serving strategies.

In [ ]:
# Create benchmark dataset
benchmark_dataset = BenchmarkDataset(
    num_samples=1000,
    input_shape=(3, 224, 224),
    num_classes=10
)

# Create data loader
data_loader = torch.utils.data.DataLoader(
    benchmark_dataset,
    batch_size=config.training.batch_size,
    shuffle=False
)

print(f"Benchmark dataset created: {len(benchmark_dataset)} samples")
print(f"Data loader created: batch size {config.training.batch_size}")

# Get a sample batch for testing
sample_batch, _ = next(iter(data_loader))
print(f"Sample batch shape: {sample_batch.shape}")

In [ ]:
# Benchmark serving adapter performance
def benchmark_adapters(adapters: Dict[str, PyTorchAdapter], 
                      data_loader: torch.utils.data.DataLoader,
                      num_batches: int = 50) -> Dict[str, Dict[str, float]]:
    """Benchmark all serving adapters."""
    
    results = {}
    
    for adapter_name, adapter in adapters.items():
        print(f"Benchmarking {adapter_name}...")
        
        # Warmup
        adapter.warmup(num_iterations=5, batch_size=config.training.batch_size)
        
        latencies = []
        throughputs = []
        
        batch_count = 0
        for batch_inputs, _ in data_loader:
            if batch_count >= num_batches:
                break
            
            # Move to device
            batch_inputs = batch_inputs.to(config.device)
            
            # Measure inference time
            start_time = time.perf_counter()
            
            try:
                predictions = adapter.predict(batch_inputs)
            except Exception as e:
                print(f"  Error in {adapter_name}: {e}")
                continue
            
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            end_time = time.perf_counter()
            
            # Calculate metrics
            inference_time = (end_time - start_time) * 1000  # ms
            batch_size = batch_inputs.shape[0]
            throughput = batch_size / (inference_time / 1000)  # samples/sec
            
            latencies.append(inference_time)
            throughputs.append(throughput)
            
            batch_count += 1
        
        if latencies:  # Only if we have valid measurements
            results[adapter_name] = {
                'mean_latency_ms': np.mean(latencies),
                'p95_latency_ms': np.percentile(latencies, 95),
                'p99_latency_ms': np.percentile(latencies, 99),
                'std_latency_ms': np.std(latencies),
                'mean_throughput': np.mean(throughputs),
                'std_throughput': np.std(throughputs),
                'num_measurements': len(latencies)
            }
        
        print(f"  Completed: {len(latencies)} measurements")
    
    return results

# Run benchmark
benchmark_results = benchmark_adapters(serving_adapters, data_loader, num_batches=20)

print(f"\nBenchmark completed for {len(benchmark_results)} adapters:")
for name, metrics in benchmark_results.items():
    print(f"  {name}:")
    print(f"    Mean Latency: {metrics['mean_latency_ms']:.2f} ± {metrics['std_latency_ms']:.2f} ms")
    print(f"    P99 Latency: {metrics['p99_latency_ms']:.2f} ms")
    print(f"    Mean Throughput: {metrics['mean_throughput']:.1f} ± {metrics['std_throughput']:.1f} samples/sec")

In [ ]:
# Visualize benchmark results
if benchmark_results:
    df_results = pd.DataFrame(benchmark_results).T
    df_results['efficiency'] = df_results['mean_throughput'] / df_results['mean_latency_ms']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Serving Adapter Performance Comparison', fontsize=16)
    
    # Latency comparison
    ax1 = axes[0, 0]
    bars1 = ax1.bar(df_results.index, df_results['mean_latency_ms'], 
                     yerr=df_results['std_latency_ms'], capsize=5, alpha=0.7)
    ax1.set_ylabel('Latency (ms)')
    ax1.set_title('Mean Inference Latency')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                 f'{height:.1f}', ha='center', va='bottom')
    
    # Throughput comparison
    ax2 = axes[0, 1]
    bars2 = ax2.bar(df_results.index, df_results['mean_throughput'],
                     yerr=df_results['std_throughput'], capsize=5, alpha=0.7)
    ax2.set_ylabel('Throughput (samples/sec)')
    ax2.set_title('Mean Throughput')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                 f'{height:.0f}', ha='center', va='bottom')
    
    # Latency distribution (P95, P99)
    ax3 = axes[1, 0]
    x = np.arange(len(df_results))
    width = 0.35
    
    bars3a = ax3.bar(x - width/2, df_results['p95_latency_ms'], width, 
                     label='P95', alpha=0.7)
    bars3b = ax3.bar(x + width/2, df_results['p99_latency_ms'], width,
                     label='P99', alpha=0.7)
    
    ax3.set_ylabel('Latency (ms)')
    ax3.set_title('Latency Percentiles')
    ax3.set_xticks(x)
    ax3.set_xticklabels(df_results.index, rotation=45, ha='right')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Efficiency scatter plot
    ax4 = axes[1, 1]
    scatter = ax4.scatter(df_results['mean_latency_ms'], df_results['mean_throughput'],
                         s=100, alpha=0.7, c=df_results['efficiency'], cmap='viridis')
    
    # Add labels for each point
    for i, adapter in enumerate(df_results.index):
        ax4.annotate(adapter.replace('_', '\n'), 
                     (df_results.iloc[i]['mean_latency_ms'], df_results.iloc[i]['mean_throughput']),
                     xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax4.set_xlabel('Mean Latency (ms)')
    ax4.set_ylabel('Mean Throughput (samples/sec)')
    ax4.set_title('Latency vs Throughput (Color = Efficiency)')
    ax4.grid(True, alpha=0.3)
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax4)
    cbar.set_label('Efficiency (throughput/latency)')
    
    plt.tight_layout()
    plt.show()
    
    # Print efficiency ranking
    print("\nEfficiency Ranking (throughput/latency):")
    efficiency_ranking = df_results.sort_values('efficiency', ascending=False)
    for i, (adapter, row) in enumerate(efficiency_ranking.iterrows(), 1):
        print(f"  {i}. {adapter}: {row['efficiency']:.3f}")
else:
    print("No benchmark results to display")

## 5. Optimization Experiment {#experiment}

Now let's run the serving strategy optimization experiment.

In [ ]:
# Create serving strategy optimizer
optimizer = ServingStrategyOptimizer(config)

# Register working adapters
working_adapters = {}
for name, adapter in serving_adapters.items():
    if name in benchmark_results:  # Only use adapters that worked in benchmark
        working_adapters[name] = adapter
        optimizer.register_serving_adapter(name, adapter)

print(f"Registered {len(working_adapters)} serving adapters for optimization:")
for name in working_adapters.keys():
    print(f"  {name}")

# Initialize metrics collector
metrics_collector = MetricsCollector(config)

print("\nOptimizer and metrics collector initialized!")

In [ ]:
# Run optimization experiment
def run_optimization_experiment(optimizer: ServingStrategyOptimizer,
                               adapters: Dict[str, PyTorchAdapter],
                               data_loader: torch.utils.data.DataLoader,
                               num_experiments: int = 500) -> List[Dict[str, Any]]:
    """Run serving strategy optimization experiment."""
    
    experiment_data = []
    
    print(f"Starting optimization experiment with {num_experiments} iterations...")
    
    batch_iterator = iter(data_loader)
    
    for experiment in range(num_experiments):
        try:
            # Get next batch
            try:
                batch_inputs, batch_targets = next(batch_iterator)
            except StopIteration:
                # Reset iterator if we run out of data
                batch_iterator = iter(data_loader)
                batch_inputs, batch_targets = next(batch_iterator)
            
            # Move to device
            batch_inputs = batch_inputs.to(config.device)
            
            # Select serving strategy
            strategy_name, adapter = optimizer.select_serving_strategy()
            
            # Run inference and measure performance
            start_time = time.perf_counter()
            predictions = adapter.predict(batch_inputs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            end_time = time.perf_counter()
            
            # Calculate metrics
            inference_time = (end_time - start_time) * 1000  # ms
            batch_size = batch_inputs.shape[0]
            throughput = batch_size / (inference_time / 1000)  # samples/sec
            
            # Mock accuracy calculation (in real scenario, use ground truth)
            if isinstance(predictions, torch.Tensor):
                predicted_classes = torch.argmax(predictions, dim=1)
                accuracy = (predicted_classes == batch_targets).float().mean().item()
            else:
                predicted_classes = torch.from_numpy(predictions).argmax(dim=1)
                accuracy = (predicted_classes == batch_targets).float().mean().item()
            
            # Mock error rate
            error_rate = max(0.0, 0.05 - accuracy)
            
            # Update optimizer
            optimizer.update_strategy_performance(
                strategy_name=strategy_name,
                latency=inference_time,
                throughput=throughput,
                accuracy=accuracy,
                error_rate=error_rate
            )
            
            # Record metrics
            metrics_collector.record_performance(
                strategy_name=strategy_name,
                latency_ms=inference_time,
                throughput_fps=throughput,
                error_rate=error_rate
            )
            
            # Store experiment data
            experiment_data.append({
                'experiment': experiment,
                'strategy': strategy_name,
                'latency_ms': inference_time,
                'throughput': throughput,
                'accuracy': accuracy,
                'error_rate': error_rate,
                'timestamp': time.time()
            })
            
            # Progress reporting
            if (experiment + 1) % 100 == 0:
                print(f"  Progress: {experiment + 1}/{num_experiments} experiments completed")
                
                # Show current arm statistics
                arm_stats = optimizer.bandit.get_arm_stats()
                for arm_name, arm in arm_stats.items():
                    if arm.pulls > 0:
                        print(f"    {arm_name}: pulls={arm.pulls}, reward={arm.avg_reward:.3f}")
        
        except Exception as e:
            print(f"  Error in experiment {experiment}: {e}")
            continue
    
    return experiment_data

# Run the experiment
if working_adapters:
    experiment_data = run_optimization_experiment(
        optimizer, working_adapters, data_loader, num_experiments=200
    )
    print(f"\nOptimization experiment completed with {len(experiment_data)} data points!")
else:
    print("No working adapters available for optimization experiment")
    experiment_data = []

## 6. Results Analysis {#analysis}

Let's analyze the optimization results and visualize the learning process.

In [ ]:
# Analyze experiment results
if experiment_data:
    df_experiments = pd.DataFrame(experiment_data)
    
    print("Experiment Data Summary:")
    print(f"  Total experiments: {len(df_experiments)}")
    print(f"  Strategies tested: {df_experiments['strategy'].nunique()}")
    print(f"  Strategy distribution:")
    strategy_counts = df_experiments['strategy'].value_counts()
    for strategy, count in strategy_counts.items():
        percentage = count / len(df_experiments) * 100
        print(f"    {strategy}: {count} experiments ({percentage:.1f}%)")
    
    print(f"\n  Performance ranges:")
    print(f"    Latency: {df_experiments['latency_ms'].min():.1f} - {df_experiments['latency_ms'].max():.1f} ms")
    print(f"    Throughput: {df_experiments['throughput'].min():.1f} - {df_experiments['throughput'].max():.1f} samples/sec")
    print(f"    Accuracy: {df_experiments['accuracy'].min():.3f} - {df_experiments['accuracy'].max():.3f}")
else:
    print("No experiment data available for analysis")
    df_experiments = None

In [ ]:
# Visualize optimization results
if df_experiments is not None and len(df_experiments) > 0:
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    fig.suptitle('Optimization Experiment Analysis', fontsize=16)
    
    strategies = df_experiments['strategy'].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(strategies)))
    
    # Plot 1: Latency over time by strategy
    ax1 = axes[0, 0]
    for strategy, color in zip(strategies, colors):
        strategy_data = df_experiments[df_experiments['strategy'] == strategy]
        ax1.scatter(strategy_data['experiment'], strategy_data['latency_ms'],
                   alpha=0.6, label=strategy, c=[color], s=20)
        
        # Add trend line
        if len(strategy_data) > 10:
            z = np.polyfit(strategy_data['experiment'], strategy_data['latency_ms'], 1)
            p = np.poly1d(z)
            ax1.plot(strategy_data['experiment'], p(strategy_data['experiment']),
                    color=color, linewidth=2, alpha=0.8)
    
    ax1.set_xlabel('Experiment Number')
    ax1.set_ylabel('Latency (ms)')
    ax1.set_title('Latency Evolution by Strategy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Strategy selection frequency over time
    ax2 = axes[0, 1]
    window_size = 50
    
    for strategy, color in zip(strategies, colors):
        strategy_freq = []
        x_positions = []
        
        for i in range(window_size, len(df_experiments), 5):
            window_data = df_experiments.iloc[i-window_size:i]
            freq = len(window_data[window_data['strategy'] == strategy]) / window_size
            strategy_freq.append(freq)
            x_positions.append(i)
        
        ax2.plot(x_positions, strategy_freq, label=strategy, color=color, linewidth=2)
    
    ax2.set_xlabel('Experiment Number')
    ax2.set_ylabel('Selection Frequency')
    ax2.set_title(f'Strategy Selection Frequency (Window: {window_size})')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Performance comparison boxplots
    ax3 = axes[1, 0]
    df_experiments.boxplot(column='latency_ms', by='strategy', ax=ax3)
    ax3.set_title('Latency Distribution by Strategy')
    ax3.set_xlabel('Strategy')
    ax3.set_ylabel('Latency (ms)')
    ax3.tick_params(axis='x', rotation=45)
    
    # Plot 4: Throughput comparison
    ax4 = axes[1, 1]
    df_experiments.boxplot(column='throughput', by='strategy', ax=ax4)
    ax4.set_title('Throughput Distribution by Strategy')
    ax4.set_xlabel('Strategy')
    ax4.set_ylabel('Throughput (samples/sec)')
    ax4.tick_params(axis='x', rotation=45)
    
    # Plot 5: Learning curve (cumulative average performance)
    ax5 = axes[2, 0]
    
    # Calculate cumulative moving average for each strategy
    for strategy, color in zip(strategies, colors):
        strategy_data = df_experiments[df_experiments['strategy'] == strategy].sort_values('experiment')
        if len(strategy_data) > 0:
            cumulative_avg_latency = strategy_data['latency_ms'].expanding().mean()
            ax5.plot(strategy_data['experiment'], cumulative_avg_latency,
                    label=f'{strategy}', color=color, linewidth=2)
    
    ax5.set_xlabel('Experiment Number')
    ax5.set_ylabel('Cumulative Average Latency (ms)')
    ax5.set_title('Learning Curve - Cumulative Average Latency')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: Final performance comparison
    ax6 = axes[2, 1]
    
    # Calculate final performance metrics
    final_performance = df_experiments.groupby('strategy').agg({
        'latency_ms': ['mean', 'std'],
        'throughput': ['mean', 'std'],
        'accuracy': 'mean'
    })
    
    # Efficiency metric (throughput / latency)
    efficiency = final_performance[('throughput', 'mean')] / final_performance[('latency_ms', 'mean')]
    
    bars = ax6.bar(range(len(efficiency)), efficiency.values, 
                   color=colors[:len(efficiency)], alpha=0.7)
    ax6.set_xlabel('Strategy')
    ax6.set_ylabel('Efficiency (throughput/latency)')
    ax6.set_title('Final Efficiency Comparison')
    ax6.set_xticks(range(len(efficiency)))
    ax6.set_xticklabels(efficiency.index, rotation=45, ha='right')
    ax6.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars, efficiency.values):
        height = bar.get_height()
        ax6.text(bar.get_x() + bar.get_width()/2., height,
                 f'{value:.3f}', ha='center', va='bottom')
    
    plt.suptitle('')  # Remove automatic boxplot title
    plt.tight_layout()
    plt.show()
else:
    print("No experiment data available for visualization")

In [ ]:
# Generate final optimization report
if experiment_data:
    report = optimizer.get_optimization_report()
    
    print("\n" + "="*60)
    print("OPTIMIZATION REPORT")
    print("="*60)
    
    print(f"Best Strategy: {report['best_strategy']}")
    print(f"Best Reward: {report['best_reward']:.4f}")
    print(f"Total Experiments: {report['total_experiments']}")
    print(f"Convergence Achieved: {report['convergence_achieved']}")
    
    print("\nStrategy Performance Summary:")
    for strategy, stats in report['strategy_statistics'].items():
        print(f"  {strategy}:")
        print(f"    Pulls: {stats['pulls']}")
        print(f"    Avg Reward: {stats['avg_reward']:.4f}")
        print(f"    P95 Latency: {stats['p95_latency_ms']:.2f} ms")
        print(f"    Avg Throughput: {stats['avg_throughput']:.1f} samples/sec")
        print(f"    Error Rate: {stats['error_rate']:.4f}")
    
    if report['performance_improvements']:
        print("\nPerformance Improvements:")
        for metric, improvement in report['performance_improvements'].items():
            print(f"  {metric}: {improvement:.1f}%")
    
    print("\n" + "="*60)
else:
    print("No optimization report available")

## 7. Production Deployment Guide {#deployment}

This section provides guidance for deploying the optimized serving strategy in production.

In [ ]:
# Production deployment recommendations
def generate_deployment_recommendations(report: Dict[str, Any], 
                                       benchmark_results: Dict[str, Any]) -> Dict[str, Any]:
    """Generate production deployment recommendations."""
    
    recommendations = {
        'primary_strategy': report['best_strategy'],
        'deployment_plan': [],
        'monitoring_recommendations': [],
        'risk_mitigation': [],
        'performance_targets': {}
    }
    
    best_strategy = report['best_strategy']
    best_stats = report['strategy_statistics'][best_strategy]
    
    # Deployment plan
    recommendations['deployment_plan'].extend([
        f"1. Deploy {best_strategy} as primary serving strategy",
        "2. Implement gradual traffic migration (10%, 25%, 50%, 100%)",
        "3. Monitor performance metrics at each migration step",
        "4. Maintain fallback capability to previous strategy",
        "5. Set up automated rollback triggers"
    ])
    
    # Monitoring recommendations
    recommendations['monitoring_recommendations'].extend([
        "Monitor P95 and P99 latency continuously",
        "Track throughput and error rates",
        "Set up alerts for performance degradation",
        "Implement model drift detection",
        "Log serving strategy selection decisions"
    ])
    
    # Risk mitigation
    recommendations['risk_mitigation'].extend([
        "Implement circuit breaker pattern for failing strategies",
        "Set up automated model retraining pipeline",
        "Maintain multiple serving strategies in production",
        "Implement gradual rollout with automated rollback",
        "Regular performance benchmarking and optimization"
    ])
    
    # Performance targets
    recommendations['performance_targets'] = {
        'p95_latency_ms': best_stats['p95_latency_ms'] * 1.1,  # 10% buffer
        'min_throughput': best_stats['avg_throughput'] * 0.9,  # 10% buffer
        'max_error_rate': best_stats['error_rate'] * 2.0,      # 2x buffer
        'min_accuracy': 0.95  # Minimum acceptable accuracy
    }
    
    return recommendations

# Generate recommendations if we have results
if experiment_data and 'report' in locals():
    deployment_rec = generate_deployment_recommendations(report, benchmark_results)
    
    print("\n" + "="*60)
    print("PRODUCTION DEPLOYMENT RECOMMENDATIONS")
    print("="*60)
    
    print(f"\nPrimary Strategy: {deployment_rec['primary_strategy']}")
    
    print("\nDeployment Plan:")
    for step in deployment_rec['deployment_plan']:
        print(f"  {step}")
    
    print("\nMonitoring Recommendations:")
    for rec in deployment_rec['monitoring_recommendations']:
        print(f"  • {rec}")
    
    print("\nRisk Mitigation:")
    for risk in deployment_rec['risk_mitigation']:
        print(f"  • {risk}")
    
    print("\nPerformance Targets:")
    for target, value in deployment_rec['performance_targets'].items():
        print(f"  {target}: {value:.2f}")
    
    print("\n" + "="*60)
else:
    print("No results available for deployment recommendations")

## Summary

This notebook has demonstrated the complete workflow of the Adaptive Model Serving Optimizer:

1. **Configuration and Setup**: Initialized the system with appropriate configurations
2. **Model Adapters**: Created and compared different serving strategies
3. **Bandit Algorithms**: Explored multi-armed bandit algorithms for optimization
4. **Performance Benchmarking**: Measured baseline performance of all strategies
5. **Optimization Experiment**: Ran the adaptive serving optimization
6. **Results Analysis**: Analyzed the learning process and final performance
7. **Deployment Guidance**: Provided production deployment recommendations

### Key Insights:

- **Strategy Comparison**: Different serving strategies have distinct performance characteristics
- **Optimization Learning**: The bandit algorithms successfully learn optimal strategies over time
- **Performance Trade-offs**: There are clear trade-offs between latency, throughput, and accuracy
- **Production Readiness**: The system provides comprehensive monitoring and deployment guidance

### Next Steps:

1. Run the full training script with more experiments for production-quality results
2. Implement additional serving strategies (TensorRT, quantized models)
3. Deploy in a production environment with real traffic
4. Continuously monitor and retrain the optimization system

The Adaptive Model Serving Optimizer provides a robust foundation for intelligent model serving that continuously optimizes performance while maintaining quality guarantees.